# 공개 침수 특징으로 Colab GPU 점검

이 노트북은 FloodNet에서 로컬로 추출한 512차원 공개 특징으로 침수 판별을 재현한다. 대회 및 AI-Hub 자료는 국내 PC에 둔다. Colab에서 GPU를 선택하고 실행 전 계정의 컴퓨팅 단위 잔액과 현재 소모율을 적어 둔다. 동일한 노트북을 Kaggle 또는 로컬 Jupyter에서도 실행할 수 있다.

In [ ]:
from pathlib import Path
import json, os, platform, time
import numpy as np
import torch
device = torch.device(os.environ.get("KWATER_DEVICE", "cuda" if torch.cuda.is_available() else "cpu"))
use_cuda = device.type == "cuda"
if use_cuda:
    props = torch.cuda.get_device_properties(0)
    print(props.name, round(props.total_memory / 2**30, 2), "GiB")
print(platform.python_version(), torch.__version__, torch.version.cuda, device)

## 공개 파일 로드

프로젝트의 `public-data/floodnet-pilot/pilot-features.npz`를 사용한다. Colab 파일 패널에 이 파일 하나를 업로드하거나 `FLOODNET_FEATURES`에 현지 파일 경로를 지정한다. 이 파일은 고정 분할·정답·공개 이미지 특징·집계 메타데이터를 담는다.

In [ ]:
options = [os.environ.get("FLOODNET_FEATURES"), "/content/pilot-features.npz",
           "public-data/floodnet-pilot/pilot-features.npz", "data/external/floodnet/pilot-features.npz", "pilot-features.npz"]
source = next((Path(p) for p in options if p and Path(p).is_file()), None)
if source is None:
    raise FileNotFoundError("Upload the public FloodNet pilot-features.npz using the notebook file panel.")
with np.load(source, allow_pickle=False) as a:
    if set(a.files) != {"sig", "features", "truth", "split", "audit"}:
        raise ValueError("Unexpected feature archive")
    x = torch.as_tensor(a["features"].copy(), dtype=torch.float32, device=device)
    y = torch.as_tensor(a["truth"].copy(), dtype=torch.float32, device=device)
    split = torch.as_tensor(a["split"].copy(), device=device)
    provenance = json.loads(a["audit"].item())
assert x.ndim == 2 and x.shape[1] == 512 and len(y) == len(x) == len(split)
assert bool(torch.isfinite(x).all()) and set(split.cpu().tolist()) == {0, 1}
xtrain, ytrain = x[split == 0], y[split == 0]
xtest, ytest = x[split == 1], y[split == 1]
print(provenance["train"], provenance["test"])

In [ ]:
def report(name, pred):
    true = ytest.bool()
    tp = int((pred & true).sum())
    fp = int((pred & ~true).sum())
    fn = int((~pred & true).sum())
    tn = int((~pred & ~true).sum())
    return {"method": name, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "F1": 2*tp / max(1, 2*tp + fp + fn)}
def center(rows):
    return torch.nn.functional.normalize(rows.mean(0), dim=0)
positive = ytrain.bool()
direction = center(xtrain[positive]) - center(xtrain[~positive])
metrics = [report("train centroids", xtest @ direction >= 0)]
def synchronize():
    if use_cuda:
        torch.cuda.synchronize(device)
torch.manual_seed(42)
if use_cuda:
    torch.cuda.reset_peak_memory_stats(device)
synchronize()
start = time.perf_counter()
# Fit only on the original training split, with no test-based hyperparameter selection.
model = torch.nn.Linear(x.shape[1], 1).to(device)
torch.nn.init.zeros_(model.weight)
torch.nn.init.zeros_(model.bias)
optim = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.01)
for _ in range(100):
    optim.zero_grad()
    logits = model(xtrain).squeeze(-1)
    loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, ytrain)
    loss.backward()
    optim.step()
synchronize()
elapsed = time.perf_counter() - start
with torch.inference_mode():
    metrics.append(report("linear head, 100 updates", model(xtest).squeeze(-1) >= 0))
result = {"device": str(device), "wall_seconds_100_updates": elapsed,
          "peak_cuda_allocated_mib": torch.cuda.max_memory_allocated(device)/2**20 if use_cuda else None,
          "selection_sha256": provenance["selection_sha256"], "metrics": metrics}
print(json.dumps(result, indent=2))
Path("floodnet-public-benchmark.json").write_text(json.dumps(result, indent=2), encoding="utf-8")

## 비용과 시간 기록

실행 뒤 Colab의 잔액 차이와 세션 시간을 적고 런타임을 연결 해제한다. 이 작은 분류기의 소요시간은 GPU 작동 여부와 재현성 확인에 쓰인다. 제출 후보의 시간 예산은 별도로 L4에서 같은 이미지 해상도, 출력 길이, 오프라인 설치 경로를 포함해 측정한다. FloodNet의 출처는 [FloodNet 연구진](https://github.com/BinaLab/FloodNet-Supervised_v1.0)이다.